# The best way to run this file

## 1. Prerequisites

Before running the code, ensure your environment meets the following requirements:

- **Python Version**: Python 3.9 or higher is recommended
- **Environment**:
  - **Google Colab** (Recommended for free GPU/RAM access) — No local installation needed
  - **Local**: Use an IDE like VS Code, PyCharm, or a classic Jupyter Notebook
- **Hardware note**:  
  Due to **10-fold nested cross-validation** across **16 datasets** and **5 models**, a full run can take **12–24 hours**.  
  Make sure your machine has a **stable power supply** and **reliable internet connection**.

## 2. Required Libraries

Install the following packages (most are already pre-installed in Colab):

```bash
!pip install pandas numpy scikit-learn joblib
```


## 3. Global Configuration {#@title}

The first code cell in this notebook is the **Experiment Controller**.

It lets you easily change the scope of the execution **without touching any of the model training or evaluation logic** (which is long).

You can:

- Run **only one model** on **only one dataset** to test / debug quickly  
- Run **only one model** on **All dataset** to see results of a single model
- Run **All** Models on **All** Datasets when you're ready for the full experiment

```python
SELECTED_MODEL = "All"   # @param ["All", "SVM", "RandomForest", "XGBoost", "MLP", "LogisticRegression"]
DATASET_TO_RUN  = "All"  # @param ["All", "Data1", "Data2", "Data3", "Data4", "Data5", "Data6", "Data7", "Data8", "Data9", "Data10", "Data11", "Data12", "Data13", "Data14", "Data15", "Data16"]
```

## 4. Generic Dataset Paths (Colab vs. Local)

To make this code run for anyone, we use a Dynamic Path Generator. You do not need to manually type 32 file paths.

**If you are running this on Google Colab:**  
Ensure your Drive is mounted first:

```python
from google.colab import drive
drive.mount('/content/drive')
```

**If you are running this on your Local Machine:**  
If you are not using Colab, change the base directory variable. The code uses a List Comprehension to build paths:

```python
# Change this base path to your local folder
BASE_PATH = "/your/local/folder/Datasets"

all_datasets = [
    (f"Data{i}",
     f"{BASE_PATH}/{i}/train.csv",
     f"{BASE_PATH}/{i}/test.csv")
    for i in range(1, 17)
]
```

## 5. Execution Steps

1. Run the **mount drive** cell (Colab only)  
2. Adjust `SELECTED_MODEL` and `DATASET_TO_RUN` in the controller cell  
3. Run all cells in order  
4. Have some coffee or even better lunch or dinner — especially with `"All" / "All"` selected ☕😄

Happy experimenting!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import + Config

In [ ]:
# ============================================
# Imports, global settings, helper functions
# ============================================
import os
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, GridSearchCV, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, f1_score, make_scorer


RANDOM_STATE = 42

# @title Global Configuration
SELECTED_MODEL = "All" # @param ["All", "SVM", "KNN", "DT", "RF", "MLP"]

BASE_PATH = "/content/drive/MyDrive/CS-6745-MLDM/Datasets"

DATASET_TO_RUN = "All" # @param ["All", "Data1", "Data2", "Data3", "Data4", "Data5", "Data6", "Data7", "Data8", "Data9", "Data10", "Data11", "Data12", "Data13", "Data14", "Data15", "Data16"]

# Dynamically generate all 16 dataset paths
all_datasets = [
    (f"Data{i}",
     os.path.join(BASE_PATH, str(i), "train.csv"),
     os.path.join(BASE_PATH, str(i), "test.csv"))
    for i in range(1, 17)
]

# Quick verification print
print(f"Generated {len(all_datasets)} dataset paths.")
print(f"Sample path: {all_datasets[0][1]}")

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def params_to_str(best_params: dict) -> str:
    """Convert GridSearchCV best_params_ dict to a compact string."""
    items = []
    for k, v in best_params.items():
        clean_k = k.replace("clf__", "").replace("pca__", "pca.")
        items.append(f"{clean_k}={v}")
    return ", ".join(items)


def dataset_code_from_name(dataset_name: str) -> str:
    """Turn 'Data1' into 'D1', etc."""
    if dataset_name.startswith("Data"):
        return "D" + dataset_name[4:]
    return dataset_name


def fold_row_template_phase1() -> dict:
    return {
        "SVM-Accuracy": "", "SVM-F1": "",
        "KNN-Accuracy": "", "KNN-F1": "",
        "DT-Accuracy": "", "DT-F1": "",
        "RF-Accuracy": "", "RF-F1": "",
        "MLP-Accuracy": "", "MLP-F1": "",
        "SVM Parameters": "", "KNN Parameters": "",
        "DT Parameters": "", "RF Parameters": "",
        "MLP-Parameters": "",
    }


def fold_row_template_phase2() -> dict:
    return {
        "SVM-Accuracy": "", "SVM-F1": "",
        "KNN-Accuracy": "", "KNN-F1": "",
        "DT-Accuracy": "", "DT-F1": "",
        "RF-Accuracy": "", "RF-F1": "",
        "MLP-Accuracy": "", "MLP-F1": "",
        "No. Selected Features": "",
        "Features Name": "",
        "FS/DS Parameters (SVM)": "",
        "FS/DS Parameters (KNN)": "",
        "FS/DS Parameters (DT)": "",
        "FS/DS Parameters (RF)": "",
        "FS/DS Parameters (MLP)": "",
    }


Generated 16 dataset paths.
Sample path: /content/drive/MyDrive/CS-6745-MLDM/Datasets/1/train.csv


# Cross Validation x HyperParam

In [ ]:
# ============================================
# CV objects and hyperparameter grids
# (Respects: 10-fold outer CV + inner CV with 10% validation) [Instructions-6735.pdf]
# ============================================

f1_macro = make_scorer(f1_score, average="macro")

# Outer loop: 10-fold CV (required)
outer_cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Inner loop: CV with 10% validation (grid search in inner loop is required)
inner_cv = StratifiedShuffleSplit(
    n_splits=3,          # still CV, but fewer splits for speed
    test_size=0.10,
    random_state=RANDOM_STATE,
)

# "Fast but valid" grids: more than one value where reasonable, but small
models = {
    "SVM": {
        "estimator": SVC(),
        "param_grid": {
            "clf__C": [1, 10],            # same
            "clf__kernel": ["rbf","sigmoid"],        # same
            "clf__gamma": ["scale", 0.01], # same
        },
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "param_grid": {
            "clf__n_neighbors": [3, 5, 7], # added 3 (was [5,7])
            "clf__metric": ["euclidean", "manhattan"],  # same
        },
    },
    "DT": {
        "estimator": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "clf__max_depth": [None, 10],  # same
            "clf__min_samples_split": [2, 5], # same
        },
    },
    "RF": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "clf__n_estimators": [100,200],    # same
            "clf__max_depth": [None, 10],  # same
        },
    },
    "MLP": {
        "estimator": MLPClassifier(max_iter=800, random_state=RANDOM_STATE),
        "param_grid": {
            "clf__hidden_layer_sizes": [(50,), (100,)], # added (100,) (was just (50,))
            "clf__learning_rate_init": [0.001,0.01],        # same
            "clf__alpha": [0.0001, 0.001],             # same
        },
    },
}

# Update active_models logic
if SELECTED_MODEL == "All":
    active_models = models_library # Use the whole library
else:
    active_models = {SELECTED_MODEL: models_library[SELECTED_MODEL]}


# Helpers / Records

In [ ]:
# ============================================
# Common helper: save one pickle per dataset × phase (D1P1.pkl, D1P2.pkl)
# ============================================

def save_phase_bundle_pickle(out_dir: str,
                             dataset_name: str,
                             dataset_code: str,
                             phase_name: str,
                             phase_code: str,
                             folds_dict: dict,
                             phase_level_meta: dict = None):
    """
    Save ONE pickle per dataset × phase, named like D1P1.pkl.

    folds_dict:
      {
        fold_id: {
          "fold_meta": {...},
          "models": {
            "SVM": {"best_params": ..., "accuracy": ..., "f1_macro": ..., "estimator": ...},
            ...
          }
        },
        ...
      }
    """
    ensure_dir(out_dir)

    tag = f"{dataset_code}{phase_code}"
    pkl_filename = f"{tag}.pkl"
    pkl_path = os.path.join(out_dir, pkl_filename)

    payload = {
        "dataset": dataset_name,
        "dataset_code": dataset_code,
        "phase": phase_name,
        "phase_code": phase_code,
        "tag": tag,
        "phase_meta": phase_level_meta if phase_level_meta is not None else {},
        "folds": folds_dict,
    }

    joblib.dump(payload, pkl_path)
    return pkl_path


In [ ]:
# ============================================
# Helpers for report: best hyperparameters, CV mean±std, test mean±std
# ============================================

def best_hyperparams_from_folds(folds_dict: dict, clf_name: str) -> str:
    """
    Decide 'best hyperparameters' for a classifier in one phase
    by taking the most frequent best_params across the 10 outer folds.
    """
    param_strs = []
    for fold_data in folds_dict.values():
        mdl = fold_data["models"][clf_name]
        param_strs.append(params_to_str(mdl["best_params"]))
    most_common = Counter(param_strs).most_common(1)[0][0]
    return most_common


def mean_std_summary(results_df: pd.DataFrame, clf_name: str):
    """
    Return formatted mean ± std strings for Accuracy and F1 across folds (CV).
    """
    acc = results_df[f"{clf_name}-Accuracy"].astype(float)
    f1v = results_df[f"{clf_name}-F1"].astype(float)

    acc_mean, acc_std = acc.mean(), acc.std(ddof=1)
    f1_mean, f1_std = f1v.mean(), f1v.std(ddof=1)

    acc_txt = f"{acc_mean:.3f} ± {acc_std:.3f}"
    f1_txt = f"{f1_mean:.3f} ± {f1_std:.3f}"
    return acc_txt, f1_txt


In [ ]:
def build_final_model_and_eval_on_test(dataset_name: str,
                                       phase_label: str,
                                       clf_name: str,
                                       best_params_str: str,
                                       X_train: pd.DataFrame,
                                       y_train: pd.Series,
                                       X_test: pd.DataFrame,
                                       y_test: pd.Series,
                                       use_pca: bool,
                                       pca_variance: float = 0.95):
    """
    Build one final pipeline using 'best_params_str',
    train on full training set, evaluate once on test set.
    Works with hyperparams_summary-2.csv format, including hidden_layer_sizes=(100,).
    """
    base_est = models[clf_name]["estimator"]

    param_dict = {}
    s = best_params_str.strip()

    # Special handling for hidden_layer_sizes so we keep the full tuple "(100,)" together
    hls_value = None
    if "hidden_layer_sizes=" in s:
        left, right = s.split("hidden_layer_sizes=", 1)
        # right is like "(100,), learning_rate_init=0.001"
        # take up to the first closing parenthesis
        closing_idx = right.find(")")
        hls_str = right[:closing_idx + 1]   # e.g. "(100,)"
        hls_value = eval(hls_str)           # tuple
        # reconstruct the string WITHOUT the hidden_layer_sizes part
        # so we can parse the remaining params with simple splitting
        after = right[closing_idx + 1:]     # ", learning_rate_init=0.001"
        s = (left + after).strip().strip(",")  # "alpha=0.001, learning_rate_init=0.001" etc.

    # Now parse the remaining "k=v" pairs (none of them contain parentheses)
    if s:
        parts = [p.strip() for p in s.split(",")]
        for p in parts:
            if not p or "=" not in p:
                continue
            k, v = p.split("=", 1)
            k = k.strip()
            v = v.strip()
            full_key = f"clf__{k}" if not k.startswith("clf__") else k

            if v in ["rbf", "linear", "euclidean", "manhattan", "scale", "auto"]:
                param_dict[full_key] = v
            elif v in ["None"]:
                param_dict[full_key] = None
            else:
                try:
                    if "." in v:
                        param_dict[full_key] = float(v)
                    else:
                        param_dict[full_key] = int(v)
                except ValueError:
                    param_dict[full_key] = v

    # Finally, if we had a hidden_layer_sizes, set that explicitly
    if hls_value is not None:
        param_dict["clf__hidden_layer_sizes"] = hls_value

    steps = [("imputer", SimpleImputer(strategy="median")),
             ("scaler", StandardScaler())]
    if use_pca:
        steps.append(("pca", PCA(n_components=pca_variance)))
    steps.append(("clf", base_est))

    pipe = Pipeline(steps=steps)
    pipe.set_params(**param_dict)

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average="macro")

    print(f"[TEST final] {dataset_name} | {phase_label} | {clf_name}: "
          f"Acc={acc:.4f}, F1={f1m:.4f}")

    return acc, f1m


# Phase 1

In [ ]:
# ============================================
# Phase 1: nested CV, per-phase CSV (D1P1.csv), phase-level pickle
# ============================================

def run_phase1_nested_cv(dataset_name: str, X: pd.DataFrame, y: pd.Series,
                         out_root: str = "/content/models"):
    phase_name = "PHASE1_BEFORE"
    phase_code = "P1"

    dataset_code = dataset_code_from_name(dataset_name)
    base_dir = os.path.join(out_root, dataset_name, "phase1_before")
    ensure_dir(base_dir)

    fold_rows = {}
    folds_dict = {}

    for fold_id, (tr_idx, va_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        row = fold_row_template_phase1()
        fold_models = {}

        for model_name, cfg in active_models.items():
            pipe = Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("clf", cfg["estimator"]),
                ]
            )

            grid = GridSearchCV(
                estimator=pipe,
                param_grid=cfg["param_grid"],
                scoring={"acc": "accuracy", "f1_macro": f1_macro},
                refit="f1_macro",
                cv=inner_cv,
                n_jobs=-1,
                return_train_score=False,
            )

            grid.fit(X_tr, y_tr)
            best_model = grid.best_estimator_

            y_pred = best_model.predict(X_va)
            acc = accuracy_score(y_va, y_pred)
            f1m = f1_score(y_va, y_pred, average="macro")

            row[f"{model_name}-Accuracy"] = f"{acc:.4f}"
            row[f"{model_name}-F1"] = f"{f1m:.4f}"
            row_key = f"{model_name} Parameters" if model_name != "MLP" else "MLP-Parameters"
            row[row_key] = params_to_str(grid.best_params_)

            fold_models[model_name] = {
                "model_name": model_name,
                "best_params": grid.best_params_,
                "accuracy": acc,
                "f1_macro": f1m,
                "estimator": best_model,
            }

        folds_dict[fold_id] = {
            "fold_meta": {},
            "models": fold_models,
        }
        fold_rows[fold_id] = row

        print(f"\n=== {dataset_name} PHASE 1 | Fold {fold_id} row ===")
        print(row)

    phase_pkl_path = save_phase_bundle_pickle(
        out_dir=base_dir,
        dataset_name=dataset_name,
        dataset_code=dataset_code,
        phase_name=phase_name,
        phase_code=phase_code,
        folds_dict=folds_dict,
        phase_level_meta={},
    )
    print("\nSaved Phase 1 bundle pickle to:", phase_pkl_path)

    # ONE CSV for all 10 folds: D1P1.csv
    results_df = pd.DataFrame.from_dict(fold_rows, orient="index")
    results_df.index.name = "Fold"
    phase_tag = f"{dataset_code}{phase_code}"  # D1P1
    results_csv = os.path.join(base_dir, f"{phase_tag}.csv")
    results_df.to_csv(results_csv)
    print("Saved Phase 1 all-folds CSV to:", results_csv)

    return results_df, folds_dict


# Phase 2

In [ ]:
# ============================================
# Phase 2 (PCA): nested CV, per-phase CSV (D1P2.csv), phase-level pickle
# ============================================

def run_phase2_pca_nested_cv(dataset_name: str, X: pd.DataFrame, y: pd.Series,
                             out_root: str = "/content/models",
                             pca_variance: float = 0.95):
    phase_name = "PHASE2_AFTER_PCA"
    phase_code = "P2"

    dataset_code = dataset_code_from_name(dataset_name)
    base_dir = os.path.join(out_root, dataset_name, "phase2_after_pca")
    ensure_dir(base_dir)

    fold_rows = {}
    folds_dict = {}

    for fold_id, (tr_idx, va_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        row = fold_row_template_phase2()
        fold_models = {}

        # PCA fit on outer-train for reporting k, features
        preproc_for_reporting = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_variance)),
            ]
        )
        preproc_for_reporting.fit(X_tr, y_tr)
        k = int(preproc_for_reporting.named_steps["pca"].n_components_)
        feature_names = [f"PC{i}" for i in range(1, k + 1)]

        row["No. Selected Features"] = str(k)
        row["Features Name"] = ",".join(feature_names)

        for model_name, cfg in active_models.items():
            pipe = Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("pca", PCA(n_components=pca_variance)),
                    ("clf", cfg["estimator"]),
                ]
            )

            grid = GridSearchCV(
                estimator=pipe,
                param_grid=cfg["param_grid"],
                scoring={"acc": "accuracy", "f1_macro": f1_macro},
                refit="f1_macro",
                cv=inner_cv,
                n_jobs=-1,
                return_train_score=False,
            )

            grid.fit(X_tr, y_tr)
            best_model = grid.best_estimator_

            y_pred = best_model.predict(X_va)
            acc = accuracy_score(y_va, y_pred)
            f1m = f1_score(y_va, y_pred, average="macro")

            row[f"{model_name}-Accuracy"] = f"{acc:.4f}"
            row[f"{model_name}-F1"] = f"{f1m:.4f}"

            combined = f"{params_to_str(grid.best_params_)}; PCA(n_components={pca_variance}, k={k})"
            row[f"FS/DS Parameters ({model_name})"] = combined

            fold_models[model_name] = {
                "model_name": model_name,
                "best_params": grid.best_params_,
                "accuracy": acc,
                "f1_macro": f1m,
                "estimator": best_model,
            }

        folds_dict[fold_id] = {
            "fold_meta": {
                "pca_k_components": k,
                "feature_names": feature_names,
            },
            "models": fold_models,
        }
        fold_rows[fold_id] = row

        print(f"\n=== {dataset_name} PHASE 2 (PCA) | Fold {fold_id} row ===")
        print(row)

    phase_pkl_path = save_phase_bundle_pickle(
        out_dir=base_dir,
        dataset_name=dataset_name,
        dataset_code=dataset_code,
        phase_name=phase_name,
        phase_code=phase_code,
        folds_dict=folds_dict,
        phase_level_meta={"pca_n_components_setting": pca_variance},
    )
    print("\nSaved Phase 2 bundle pickle to:", phase_pkl_path)

    results_df = pd.DataFrame.from_dict(fold_rows, orient="index")
    results_df.index.name = "Fold"
    phase_tag = f"{dataset_code}{phase_code}"  # D1P2
    results_csv = os.path.join(base_dir, f"{phase_tag}.csv")
    results_df.to_csv(results_csv)
    print("Saved Phase 2 all-folds CSV to:", results_csv)

    return results_df, folds_dict


# I/O Loop with Summaries

In [ ]:
# ============================================
# PART 6: MASTER REPORTING (TABLES 1, 2, 3, & 4)
# ============================================

# Storage for Master Tables 3 and 4
all_best_hp_rows = []   # For Table 3
all_mean_std_rows = []  # For Table 4

# Storage for Master Tables 1 and 2 (Fold-by-Fold)
all_folds_before_pca = [] # For Table 1
all_folds_after_pca = []  # For Table 2

if DATASET_TO_RUN == "All":
    target_list = all_datasets
else:
    target_list = [d for d in all_datasets if d[0] == DATASET_TO_RUN]

for name, tr_path, te_path in target_list:
    print("\n" + "="*60)
    print(f"### PROCESSING: {name} | MODELS: {list(active_models.keys())}")
    print("="*60)

    # Load Data
    try:
        df_tr = pd.read_csv(tr_path)
        df_te = pd.read_csv(te_path)
        X_train, y_train = df_tr.drop(columns=["Label"]), df_tr["Label"]
        X_test, y_test = df_te.drop(columns=["Label"]), df_te["Label"]
    except Exception as e:
        print(f"Skipping {name}: {e}")
        continue

    # --- EXECUTION ---
    p1_df, p1_folds = run_phase1_nested_cv(dataset_name=name, X=X_train, y=y_train)
    p2_df, p2_folds = run_phase2_pca_nested_cv(dataset_name=name, X=X_train, y=y_train, pca_variance=0.95)

    # --- PROCESSING FOR TABLES ---
    for phase_label, df_res, folds, master_fold_list, use_pca in [
        ("Before", p1_df, p1_folds, all_folds_before_pca, False),
        ("After", p2_df, p2_folds, all_folds_after_pca, True)
    ]:
        # A) TABLE 1 & 2: Accumulate Fold Results
        df_temp = df_res.copy()
        df_temp.insert(0, 'Dataset', name)
        master_fold_list.append(df_temp)

        # Create row for Table 4
        row_ms = {"Dataset": name, "Phase": phase_label}

        for clf in active_models.keys():
            # B) TABLE 3: Best Hyperparameters
            best_hp = best_hyperparams_from_folds(folds, clf)
            all_best_hp_rows.append({
                "Classifier": clf,
                "Phase": phase_label,
                "Dataset": name,
                "Best Hyperparameters": best_hp
            })

            # C) EVALUATE ON TEST SET (Section 11)
            # This ensures we still get that final verification score
            t_acc, t_f1 = build_final_model_and_eval_on_test(
                dataset_name=name, phase_label=phase_label, clf_name=clf,
                best_params_str=best_hp, X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test, use_pca=use_pca, pca_variance=0.95
            )

            # D) TABLE 4: Mean ± Std (Validation) + Final Test Score
            acc_txt, f1_txt = mean_std_summary(df_res, clf)
            row_ms[clf] = f"CV: {acc_txt} | Test: {t_acc:.4f}"

        all_mean_std_rows.append(row_ms)

# ============================================
# FINAL EXPORT TO CSV
# ============================================
summary_dir = "/content/models/_summaries"
ensure_dir(summary_dir)
suffix = f"_{SELECTED_MODEL}_{DATASET_TO_RUN}"

if all_folds_before_pca:
    pd.concat(all_folds_before_pca).to_csv(f"{summary_dir}/Table1_Folds_Before{suffix}.csv")
if all_folds_after_pca:
    pd.concat(all_folds_after_pca).to_csv(f"{summary_dir}/Table2_Folds_After{suffix}.csv")

pd.DataFrame(all_best_hp_rows).to_csv(f"{summary_dir}/Table3_Best_HP{suffix}.csv", index=False)
pd.DataFrame(all_mean_std_rows).to_csv(f"{summary_dir}/Table4_Final_Results{suffix}.csv", index=False)

print("\n" + "="*40)
print("COMPLETED: All 4 Master Tables are ready in _summaries")
print("="*40)

In [ ]:
# # ============================================
# # DATASETS LOOP + summaries
# # - hyperparams_summary.csv
# # - optional_cv_results_mean_std.csv
# # - final_results_mean_std.csv (test mean ± std; use this in Section 9 table)
# # ============================================
# # 1. Initialize storage for this specific run
# all_hyperparam_rows = []
# all_cv_metrics_rows = []
# all_final_rows = []
# all_test_rows = []

# # 2. Filter datasets based on your selection in Part 1
# if DATASET_TO_RUN == "All":
#     target_list = all_datasets
# else:
#     # Look for the specific dataset name in your all_datasets list
#     target_list = [d for d in all_datasets if d[0] == DATASET_TO_RUN]

# if not target_list:
#     print(f"Error: Dataset '{DATASET_TO_RUN}' not found. Check your Part 1 'all_datasets' list.")

# # 3. Main Execution Loop
# for name, tr_path, te_path in target_list:
#     print("\n" + "="*60)
#     print(f"### PROCESSING DATASET: {name}")
#     print(f"### ACTIVE MODEL: {SELECTED_MODEL}")
#     print("="*60)

#     # Load Data
#     try:
#         df_tr = pd.read_csv(tr_path)
#         df_te = pd.read_csv(te_path)
#     except FileNotFoundError:
#         print(f"CRITICAL ERROR: File not found at {tr_path}. Skipping dataset.")
#         continue

#     # Prepare Features and Labels (Assumes column name is 'Label')
#     X_train = df_tr.drop(columns=["Label"])
#     y_train = df_tr["Label"]
#     X_test = df_te.drop(columns=["Label"])
#     y_test = df_te["Label"]

#     # --- PHASE 1: Baseline (No PCA) ---
#     print(f"\n--- Running Phase 1 (Baseline) ---")
#     phase1_df, phase1_folds = run_phase1_nested_cv(
#         dataset_name=name,
#         X=X_train,
#         y=y_train
#         # phase_code="P1",
#         # use_pca=False
#     )

#     # --- PHASE 2: After PCA ---
#     print(f"\n--- Running Phase 2 (PCA @ 95%) ---")
#     phase2_df, phase2_folds = run_phase2_pca_nested_cv(
#         dataset_name=name,
#         X=X_train,
#         y=y_train
#         # phase_code="P2",
#         # use_pca=True,
#         # pca_variance=0.95
#     )

#     # 4. Generate Summaries for the Current Model Selection
#     for phase_label, df_res, folds, use_pca in [
#         ("Before", phase1_df, phase1_folds, False),
#         ("After (PCA)", phase2_df, phase2_folds, True),
#     ]:
#         # We only process the one model currently selected
#         for clf in active_models.keys():

#             # A) Extract Best Hyperparameters (For Section 8 of Report)
#             best_hp = best_hyperparams_from_folds(folds, clf)
#             all_hyperparam_rows.append({
#                 "Dataset": name,
#                 "Phase": phase_label,
#                 "Classifier": clf,
#                 "Best Hyperparameters": best_hp,
#             })

#             # B) Calculate CV Mean ± Std (For Section 9 Table)
#             acc_txt_cv, f1_txt_cv = mean_std_summary(df_res, clf)
#             all_cv_metrics_rows.append({
#                 "Dataset": name,
#                 "Phase": phase_label,
#                 "Classifier": clf,
#                 "Accuracy (CV mean ± std)": acc_txt_cv,
#                 "F1 (CV mean ± std)": f1_txt_cv,
#             })

#             # C) Evaluate on the unseen test.csv (Section 11 Verification)
#             test_acc, test_f1 = build_final_model_and_eval_on_test(
#                 dataset_name=name,
#                 phase_label=phase_label,
#                 clf_name=clf,
#                 best_params_str=best_hp,
#                 X_train=X_train, y_train=y_train,
#                 X_test=X_test, y_test=y_test,
#                 use_pca=use_pca,
#                 pca_variance=0.95,
#             )

#             all_final_rows.append({
#                 "Dataset": name,
#                 "Phase": phase_label,
#                 "Classifier": clf,
#                 "Train Accuracy (mean ± std)": acc_txt_cv,
#                 "Train F1 (macro, mean ± std)": f1_txt_cv,
#                 "Test Accuracy": f"{test_acc:.4f}",
#                 "Test F1 (macro)": f"{test_f1:.4f}",
#             })

# # 5. Final Save and Status Report
# summary_dir = "/content/models/_summaries"
# ensure_dir(summary_dir)

# # Create a unique suffix to prevent overwriting other models' results
# suffix = f"_{SELECTED_MODEL}_{DATASET_TO_RUN}"
# hp_path = os.path.join(summary_dir, f"hyperparams{suffix}.csv")
# cv_path = os.path.join(summary_dir, f"cv_metrics{suffix}.csv")
# fn_path = os.path.join(summary_dir, f"final_results{suffix}.csv")

# pd.DataFrame(all_hyperparam_rows).to_csv(hp_path, index=False)
# pd.DataFrame(all_cv_metrics_rows).to_csv(cv_path, index=False)
# pd.DataFrame(all_final_rows).to_csv(fn_path, index=False)

# print("\n" + "="*40)
# print(f"EXECUTION COMPLETE FOR {SELECTED_MODEL}")
# print("="*40)
# print(f"Hyperparameters: {hp_path}")
# print(f"CV Metrics:      {cv_path}")
# print(f"Final Results:   {fn_path}")
# print("="*40)


In [ ]:
# OPTIONAL: download important outputs from Colab

# from google.colab import files

# # Example for Data1
# files.download("/content/models/Data1/phase1_before/D1P1.csv")
# files.download("/content/models/Data1/phase1_before/D1P1.pkl")
# files.download("/content/models/Data1/phase2_after_pca/D1P2.csv")
# files.download("/content/models/Data1/phase2_after_pca/D1P2.pkl")

# # Summaries for report
# files.download("/content/models/_summaries/hyperparams_summary.csv")
# files.download("/content/models/_summaries/final_results_mean_std.csv")
# files.download("/content/models/_summaries/optional_cv_results_mean_std.csv")
# files.download("/content/models/_summaries/final_results_testDataset.csv")


In [ ]:
!zip -r models_and_results.zip /content/models
from google.colab import files
files.download("models_and_results.zip")

  adding: content/models/ (stored 0%)
  adding: content/models/_summaries/ (stored 0%)
  adding: content/models/_summaries/final_results_testDataset.csv (deflated 52%)
  adding: content/models/_summaries/optional_cv_results_mean_std.csv (deflated 59%)
  adding: content/models/_summaries/hyperparams_summary.csv (deflated 60%)
  adding: content/models/_summaries/final_results_mean_std.csv (deflated 58%)
  adding: content/models/Data1/ (stored 0%)
  adding: content/models/Data1/phase1_before/ (stored 0%)
  adding: content/models/Data1/phase1_before/D1P1.pkl (deflated 80%)
  adding: content/models/Data1/phase1_before/D1P1.csv (deflated 83%)
  adding: content/models/Data1/phase2_after_pca/ (stored 0%)
  adding: content/models/Data1/phase2_after_pca/D1P2.pkl (deflated 70%)
  adding: content/models/Data1/phase2_after_pca/D1P2.csv (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>